# Caption-model comparison: BLIP-2 vs BLIP-base (Kaggle GPU)

**Run this notebook on Kaggle, not locally.** `Salesforce/blip2-opt-2.7b` is a ~15&nbsp;GB
(fp32 on disk) / ~7-8&nbsp;GB (fp16 in memory) model — too large to dev-iterate on a
16&nbsp;GB-unified-memory laptop. Kaggle's free GPU sessions (P100 16&nbsp;GB / 2xT4, 30
GPU-hrs/week, ~70&nbsp;GB scratch disk) give us the headroom this needs, at zero cost.

**What this notebook produces** (the captioning-stage exit-gate artifacts):
1. Both models captioning the *same* ~200-image sample (deterministic seed -> fair comparison)
2. A quantitative table: latency (s/image) + caption length per model
3. A qualitative side-by-side CSV for the manual review of 50 captions
4. Two `*_enriched.parquet` files (one per model) to pull back to the Mac

**Setup on Kaggle (one-time, before running cells):**
1. Create a new Notebook, enable a GPU accelerator (Settings -> Accelerator -> GPU T4 x2 or P100).
2. Upload `data/processed/products.parquet` from this repo as a Kaggle **Dataset** (New Dataset ->
   drag the file in) and attach it to this notebook (Add Input -> your dataset). It's ~26&nbsp;MB,
   well under Kaggle's per-file limits — no need to commit a binary parquet to git.
3. Run the cells top to bottom.

> **If you previously ran `pip install -r requirements.txt` in this session:** that downgraded
> `numpy` underneath Kaggle's pre-compiled `pandas`, corrupting the environment's C-ABI
> (`numpy.dtype size changed... Expected 96... got 88`). Re-running the fixed cell below
> *will not* undo this — pip already overwrote the files in place. **Restart the session**
> (Run -> Restart session, or the power-button icon) to get Kaggle's matched stack back, *then*
> run cells top to bottom. The fixed setup cell never force-installs `requirements.txt` — it
> only installs anything that's genuinely missing, with `--no-deps`, so this shouldn't recur.</cell id="cell-0">


In [ ]:
# Clone the repo (public).
#
# `rm -rf` first makes this cell idempotent: a half-finished clone from an earlier attempt
# (e.g. one that hung on a credential prompt while the repo was still private) would otherwise
# leave a stale `ShopTalk/` dir that `git clone` skips -- producing a confusing
# "ModuleNotFoundError: No module named 'src.captioning'" if `src/` exists but is missing
# subpackages added after that stale clone was made.
!rm -rf ShopTalk
!git clone --depth 1 https://github.com/dceshubh/ShopTalk.git
%cd ShopTalk

# Deliberately DO NOT `pip install -r requirements.txt` here. Kaggle's base image ships
# numpy/pandas/pyarrow/torch/pillow/transformers as one mutually-compiled stack (built
# against each other's C ABI). Forcing our pinned versions on top downgrades numpy beneath
# Kaggle's pre-compiled pandas -> "ValueError: numpy.dtype size changed... Expected 96 ...
# got 88" (a binary-incompatibility, not a Python-level version conflict).
#
# `caption.py` is written version-generically (explicit model registry + `hasattr` checks,
# not hardcoded-version assumptions -- see _MODEL_REGISTRY and the num_query_tokens fix), so
# it runs fine against whatever matched stack Kaggle already has. We only install anything
# that's actually missing, and with --no-deps so pip can't cascade-resolve its way into
# another ABI break.
import importlib
import subprocess
import sys

CHECKS = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("pyarrow", "pyarrow"),
    ("torch", "torch"),
    ("PIL", "pillow"),
    ("httpx", "httpx"),
    ("yaml", "pyyaml"),
    ("pydantic", "pydantic"),
    ("transformers", "transformers"),
]
for import_name, pip_name in CHECKS:
    try:
        mod = importlib.import_module(import_name)
        print(f"{import_name:14s} OK      {getattr(mod, '__version__', '')}")
    except ImportError:
        print(f"{import_name:14s} MISSING -- installing {pip_name} (--no-deps, to protect Kaggle's matched ABI stack)")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", pip_name], check=True)
        importlib.import_module(import_name)

In [ ]:
# Locate the uploaded products.parquet (attached as a Kaggle Dataset under /kaggle/input/)
# and copy it into the path src.common.config expects (configs/config.yaml: paths.products_parquet).
import glob
import shutil
from pathlib import Path

candidates = glob.glob("/kaggle/input/**/products.parquet", recursive=True)
assert candidates, (
    "products.parquet not found under /kaggle/input — upload it as a Kaggle Dataset "
    "(New Dataset -> drag in data/processed/products.parquet) and attach it via Add Input."
)
Path("data/processed").mkdir(parents=True, exist_ok=True)
shutil.copy(candidates[0], "data/processed/products.parquet")
print("Copied", candidates[0], "-> data/processed/products.parquet")

In [ ]:
# Caption the SAME ~200-image sample with both models. build_enriched_dataset samples with
# a fixed random_state (configs/config.yaml: dataset.random_seed), so both runs see identical
# products -- a true apples-to-apples comparison, not two different slices of the catalog.
import time

from src.captioning.enrich import build_enriched_dataset

SAMPLE_SIZE = 200
MODELS = ["Salesforce/blip2-opt-2.7b", "Salesforce/blip-image-captioning-base"]

runs = {}
for model_name in MODELS:
    short = model_name.split("/")[-1]
    t0 = time.time()
    df = build_enriched_dataset(
        sample_size=SAMPLE_SIZE,
        model_name=model_name,
        output_path=f"/kaggle/working/products_enriched_{short}.parquet",
    )
    elapsed = time.time() - t0
    runs[model_name] = {"df": df, "elapsed_s": elapsed}
    print(f"{model_name}: {elapsed:.1f}s total -> {elapsed / SAMPLE_SIZE:.2f}s/image")

In [ ]:
# Quantitative comparison table -- the "quality + speed" half of the documented-comparison exit gate.
import pandas as pd

summary = pd.DataFrame(
    {
        "model": list(runs),
        "total_seconds": [r["elapsed_s"] for r in runs.values()],
        "seconds_per_image": [r["elapsed_s"] / SAMPLE_SIZE for r in runs.values()],
        "non_empty_captions": [int(r["df"]["visual_caption"].notna().sum()) for r in runs.values()],
        "avg_caption_words": [
            r["df"]["visual_caption"].dropna().str.split().str.len().mean() for r in runs.values()
        ],
    }
)
print(summary.to_string(index=False))
summary.to_csv("/kaggle/working/caption_model_speed_quality_summary.csv", index=False)

In [ ]:
# Qualitative side-by-side: same item_id, both captions, for the manual review of 50.
# This produces the artifact a human reviews -- scoring on-topic-ness / visual-attribute
# mentions is a judgment call, not something to fake with a heuristic.
blip2_short, base_short = (m.split("/")[-1] for m in MODELS)

blip2_cols = runs[MODELS[0]]["df"][["item_id", "product_type", "name", "visual_caption"]].rename(
    columns={"visual_caption": f"caption_{blip2_short}"}
)
base_cols = runs[MODELS[1]]["df"][["item_id", "visual_caption"]].rename(
    columns={"visual_caption": f"caption_{base_short}"}
)
side_by_side = blip2_cols.merge(base_cols, on="item_id")

review_sample = side_by_side.sample(n=50, random_state=42).reset_index(drop=True)
review_sample["on_topic_blip2"] = ""        # fill in: y/n during manual review
review_sample["mentions_visual_attr_blip2"] = ""
review_sample[f"on_topic_{base_short}"] = ""
review_sample[f"mentions_visual_attr_{base_short}"] = ""
review_sample.to_csv("/kaggle/working/caption_manual_review_50.csv", index=False)
review_sample.head(10)

## Pulling results back to the Mac

From the Kaggle **Output** tab, download:
- `products_enriched_blip2-opt-2.7b.parquet`, `products_enriched_blip-image-captioning-base.parquet`
  -> place under `data/processed/` locally
- `caption_model_speed_quality_summary.csv` -> the quantitative half of the documented comparison
- `caption_manual_review_50.csv` -> open it, fill in the `on_topic_*` / `mentions_visual_attr_*`
  columns by eye for all 50 rows, and check the exit-gate threshold (>=90% on-topic & mentioning
  a visual attribute) once scored

Then write up the documented comparison (quality + speed -> chosen-model justification) and the
before/after enrichment diff for 10 products back in the main repo, per the captioning-stage
exit gate in `docs/ShopTalk_Plan.md`.